In [1]:
import os
import openai
import configparser

conf = configparser.ConfigParser()
current_directory = os.path.dirname(os.path.realpath('__file__'))
config_file_path = os.path.join(current_directory, '..', '..','config.ini')
conf.read(config_file_path)
openai.api_key = conf.get("Openai", "api_key")
os.environ["HTTP_PROXY"] = conf.get("Proxy", "HTTP_PROXY")  # 配置自己的代理
os.environ["HTTPS_PROXY"] = conf.get("Proxy", "HTTPS_PROXY")


In [2]:
all_models = openai.Model.list()
all_models = [model.get('id') for model in all_models.get("data")]
print(all_models, len(all_models))

['dall-e-3', 'whisper-1', 'tts-1', 'dall-e-2', 'tts-1-hd-1106', 'tts-1-hd', 'gpt-4-turbo-2024-04-09', 'gpt-4-turbo', 'gpt-4-0613', 'gpt-4o-2024-05-13', 'gpt-4-0125-preview', 'gpt-4', 'text-embedding-3-small', 'gpt-4-turbo-preview', 'text-embedding-3-large', 'babbage-002', 'gpt-3.5-turbo-0125', 'tts-1-1106', 'gpt-3.5-turbo', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo-instruct-0914', 'gpt-4-1106-preview', 'gpt-3.5-turbo-1106', 'text-embedding-ada-002', 'davinci-002', 'gpt-4o', 'gpt-3.5-turbo-16k'] 27


In [3]:
openai.Model.retrieve("dall-e-3")  # 获取单个模型详细信息

<Model model id=dall-e-3 at 0x114e847c0> JSON: {
  "id": "dall-e-3",
  "object": "model",
  "created": 1698785189,
  "owned_by": "system"
}

In [10]:
text_model = 'gpt-3.5-turbo-instruct'
chat_model = "gpt-3.5-turbo"

In [3]:
openai.Model.retrieve(chat_model)  # 获取单个模型详细信息

<Model model id=gpt-3.5-turbo at 0x107a48b30> JSON: {
  "id": "gpt-3.5-turbo",
  "object": "model",
  "created": 1677610602,
  "owned_by": "openai",
  "permission": [
    {
      "id": "modelperm-zy5TOjnE2zVaicIcKO9bQDgX",
      "object": "model_permission",
      "created": 1690864883,
      "allow_create_engine": false,
      "allow_sampling": true,
      "allow_logprobs": true,
      "allow_search_indices": false,
      "allow_view": true,
      "allow_fine_tuning": false,
      "organization": "*",
      "group": null,
      "is_blocking": false
    }
  ],
  "root": "gpt-3.5-turbo",
  "parent": null
}

In [4]:
openai.Model.retrieve(text_model)  # 获取单个模型详细信息

<Model model id=text-davinci-003 at 0x10a24a390> JSON: {
  "id": "text-davinci-003",
  "object": "model",
  "created": 1669599635,
  "owned_by": "openai-internal",
  "permission": [
    {
      "id": "modelperm-a6niqBmW2JaGmo0fDO7FEt1n",
      "object": "model_permission",
      "created": 1690930172,
      "allow_create_engine": false,
      "allow_sampling": true,
      "allow_logprobs": true,
      "allow_search_indices": false,
      "allow_view": true,
      "allow_fine_tuning": false,
      "organization": "*",
      "group": null,
      "is_blocking": false
    }
  ],
  "root": "text-davinci-003",
  "parent": null
}

## OpenAI上下文补全Completions与ChatCompletions API

In [6]:
# Completion
conversation = openai.Completion.create(
    model=text_model,
    prompt='You are a translation assistant',
    max_tokens=10,
    temperature=0.2,
    stream=False
)
text = conversation["choices"][0]['text']
print(text)



As a translation assistant, you would be


In [8]:
# Completion
conversation = openai.Completion.create(
    model=chat_model,
    prompt='You are a translation assistant',
    max_tokens=10,
    temperature=0.2,
    stream=False
)
text = conversation["choices"][0]['text']
print(text)

InvalidRequestError: This is a chat model and not supported in the v1/completions endpoint. Did you mean to use v1/chat/completions?

In [17]:
conversation_2 = openai.Completion.create(
    model=text_model,
    prompt=text,
    max_tokens=10,
    temperature=0.2,
    stream=False
)

In [19]:
text2 = conversation_2["choices"][0]['text']
print(text2)

 to help the translator with the translation process. You


In [36]:
# ChatCompletion
message = [
    {"role": "system", "content": "You are a translation assistant, and you name is Macy, a kindness young lady."},
    {"role": "user", "content": "Hi Macy! I am glad to see you."}
]
conversation_3 = openai.ChatCompletion.create(
    model=chat_model,
    messages=message,
    max_tokens=50,
    temperature=0.2,
    stream=True
)
for event in conversation_3:
    if event['choices'][0]['finish_reason'] != 'stop':
        print(event['choices'][0]['delta']['content'], end='')

Hello! I'm delighted to see you too. How can I assist you today?

In [12]:
# Completion
message = [
    {"role": "system", "content": "You are a translation assistant, and you name is Macy, a kindness young lady."},
    {"role": "user", "content": "Hi Macy! I am glad to see you."}
]
conversation = openai.ChatCompletion.create(
    model=chat_model,
    messages=message,
    max_tokens=100,
    temperature=0.2,
    # stream=False
)
for event in conversation:
    print(event['choices'])
    if event['choices'][0]['finish_reason'] != 'stop':
        print(event['choices'][0]['delta']['content'], end='')

TypeError: string indices must be integers